In [ ]:
# Import necessary libraries and load the data
import pandas as pd
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [6]:
df = pd.read_csv('illinois_plumbers_data.csv')

df.head(10)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1308 entries, 0 to 1307
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   License Number           1308 non-null   object 
 1   licenseGrp               0 non-null      float64
 2   licenseType              1308 non-null   object 
 3   License Inactive?        1308 non-null   object 
 4   Bond Expiration Date     1294 non-null   object 
 5   Name                     1308 non-null   object 
 6   Address                  1308 non-null   object 
 7   Phone                    1296 non-null   object 
 8   License Expiration Date  1308 non-null   object 
 9   totalRecords             0 non-null      float64
dtypes: float64(2), object(8)
memory usage: 102.3+ KB


In [ ]:
# Defining the desired columns to keep in the DataFrame
desired_columns = [
    "License Number",
    "Name",
    "Address",
    "Phone",
    "License Expiration Date",
    "Bond Expiration Date",
    "License Inactive?"
]

# Keep only the desired columns
df = df[desired_columns].copy()

# Remove duplicate records based on License Number
initial_count = len(df)
df.drop_duplicates(subset=["License Number"], inplace=True)
print(f"Removed {initial_count - len(df)} duplicate rows.")
print(f"Current row count: {len(df)}")

Removed 31 duplicate rows.
Current row count: 1277


In [7]:
# Replace null/NaN values with empty strings across the DataFrame
df.fillna("", inplace=True)

print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
License Number             0
licenseGrp                 0
licenseType                0
License Inactive?          0
Bond Expiration Date       0
Name                       0
Address                    0
Phone                      0
License Expiration Date    0
totalRecords               0
dtype: int64


/tmp/ipykernel_8598/1677973128.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)


In [16]:
# Function to format addresses into street, city, state, and zip code
def parse_address_robust(address_str):
    if not isinstance(address_str, str) or not address_str.strip():
        return pd.Series(["", "", "IL", ""])
    
    clean_addr = address_str.strip()
    
    # Look for a 5-digit zip code at the end of the string
    zip_match = re.search(r'\b(\d{5}(-\d{4})?)\s*$', clean_addr)
    zip_code = zip_match.group(1) if zip_match else ""
    
    # Remove the zip code from the working string
    if zip_code:
        clean_addr = clean_addr.replace(zip_code, "").strip()
        
    # Look for state (IL or ILLINOIS) near the end
    state_match = re.search(r'\b(IL|ILLINOIS)\b\.?\s*$', clean_addr, re.IGNORECASE)
    state = "IL"
    if state_match:
        clean_addr = clean_addr.replace(state_match.group(0), "").strip()
        
    # Whatever is left at the end is usually the City (e.g., 'CHICAGO')
    # Let's split by common patterns or take the last word/tokens before state
    parts = clean_addr.split()
    if len(parts) > 1:
        # Assuming the last token(s) before state/zip represent the city
        city = parts[-1].strip(",")
        street = " ".join(parts[:-1]).strip(",")
    else:
        city = clean_addr
        street = ""
        
    return pd.Series([street, city, state, zip_code])

# Test it on your DataFrame
parsed_addresses = df["Address"].apply(parse_address_robust)
parsed_addresses.columns = ["Street Address", "City", "State", "Zip Code"]

# Preview the results to verify
preview_df = pd.concat([df["Address"], parsed_addresses], axis=1)
display(preview_df.head(10))

,Address,Street Address,City,State,Zip Code
0,"4137 W MELROSE ST, CHICAGO, IL 60641",4137 W MELROSE ST,CHICAGO,IL,60641
1,"9325 OSCEOLA AVE, MORTON GROVE, IL 60053","9325 OSCEOLA AVE, MORTON",GROVE,IL,60053
2,"3336 S LOMBARD, BERWYN, IL 60402",3336 S LOMBARD,BERWYN,IL,60402
3,"4532 W SCHUBERT AVE, CHICAGO, IL 60639",4532 W SCHUBERT AVE,CHICAGO,IL,60639
4,"1580 N NORTHWEST HIGHWAY, PARK RIDGE, IL","1580 N NORTHWEST HIGHWAY, PARK",RIDGE,IL,
5,"5033 N NASHVILLE AVENUE, CHICAGO, IL 60656",5033 N NASHVLE AVENUE,CHICAGO,IL,60656
6,"5105 N TRIPP AVE, CHICAGO, IL 60630",5105 N TRIPP AVE,CHICAGO,IL,60630
7,"PO BOX 45, NEW LENOX, IL 60451","PO BOX 45, NEW",LENOX,IL,60451
8,"330 BOND STREET, ELK GROVE VILLAGE, IL 60007","330 BOND STREET, ELK GROVE",VLAGE,IL,60007
9,"330 BOND ST, ELK GROVE VILLAGE, IL 60047","330 BOND ST, ELK GROVE",VLAGE,IL,60047


In [13]:
# Function to format phone numbers into a standard format
def format_phone(phone_str):
    if not isinstance(phone_str, str) or not phone_str.strip():
        return ""
    
    # Extract only digits
    digits = re.sub(r'\D', '', phone_str)
    
    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"
    elif len(digits) == 11 and digits.startswith("1"):
        return f"({digits[1:4]}) {digits[4:7]}-{digits[7:]}"
        
    return phone_str.strip()

df["Phone"] = df["Phone"].apply(format_phone)

print("Phone number sample:")
display(df[["Phone"]].head(5))

Phone number sample:


,Phone
0,(773) 971-1018
1,(773) 988-7602
2,(708) 699-2407
3,(773) 575-0308
4,


In [17]:
# 1. Apply the comma-splitting parser to get the new address columns
parsed_addresses = df["Address"].apply(parse_address_robust)
parsed_addresses.columns = ["Street Address", "City", "State", "Zip Code"]

# 2. Join the new columns into your active DataFrame
df = pd.concat([df, parsed_addresses], axis=1)

# 3. Final arrangement of columns (now including the new columns)
final_columns = [
    "License Number",
    "Name",
    "Street Address",
    "City",
    "State",
    "Zip Code",
    "Phone",
    "License Expiration Date",
    "Bond Expiration Date",
    "License Inactive?"
]

df_cleaned = df[final_columns].copy()

# 4. Save output to file
output_filename = "illinois_plumbers_cleaned.csv"
df_cleaned.to_csv(output_filename, index=False)

print(f"Data cleaning complete! Saved {len(df_cleaned)} records to {output_filename}")
display(df_cleaned.head(10))

Data cleaning complete! Saved 1308 records to illinois_plumbers_cleaned.csv


,License Number,Name,Street Address,Street Address,City,City,State,State,Zip Code,Zip Code,Phone,License Expiration Date,Bond Expiration Date,License Inactive?
0,SBC201047,007 PLUMBING,"4137 W MELROSE ST, CHICAGO, IL 60641",4137 W MELROSE ST,,CHICAGO,IL,IL,,60641,(773) 971-1018,05/18/2027,04/13/2027,No
1,SBC198135,20/20 PLUMBING INC,"9325 OSCEOLA AVE, MORTON GROVE, IL 60053","9325 OSCEOLA AVE, MORTON",,GROVE,IL,IL,,60053,(773) 988-7602,05/05/2027,05/04/2027,No
2,BC186627,"24/7 PLUMBING & SEWER, LLC","3336 S LOMBARD, BERWYN, IL 60402",3336 S LOMBARD,,BERWYN,IL,IL,,60402,(708) 699-2407,04/12/2024,04/29/2023,Yes
3,BC209990,606 NORTHSIDE PLUMBING CORP.,"4532 W SCHUBERT AVE, CHICAGO, IL 60639",4532 W SCHUBERT AVE,,CHICAGO,IL,IL,,60639,(773) 575-0308,02/11/2027,12/05/2026,No
4,BC14690,911 PLUMBING INC,"1580 N NORTHWEST HIGHWAY, PARK RIDGE, IL","1580 N NORTHWEST HIGHWAY, PARK",,RIDGE,IL,IL,,,,11/09/2026,09/30/2026,No
5,SBC196848,"99 PLUMBING, INC.","5033 N NASHVILLE AVENUE, CHICAGO, IL 60656",5033 N NASHVLE AVENUE,,CHICAGO,IL,IL,,60656,(773) 255-3776,09/11/2026,08/27/2026,No
6,SBC186529,"A & C PLUMBING SERVICES, INC.","5105 N TRIPP AVE, CHICAGO, IL 60630",5105 N TRIPP AVE,,CHICAGO,IL,IL,,60630,(773) 736-5105,04/03/2027,02/12/2027,No
7,BC16837,"A & D PLUMBING, INC.","PO BOX 45, NEW LENOX, IL 60451","PO BOX 45, NEW",,LENOX,IL,IL,,60451,(815) 726-1004,04/30/2024,03/26/2023,Yes
8,BC210657,A & H PLUMBING & HEATING CO IN,"330 BOND STREET, ELK GROVE VILLAGE, IL 60007","330 BOND STREET, ELK GROVE",,VLAGE,IL,IL,,60007,(847) 981-8800,06/06/2027,06/03/2027,No
9,BC183977,"A & H PLUMBING & HEATING,","330 BOND ST, ELK GROVE VILLAGE, IL 60047","330 BOND ST, ELK GROVE",,VLAGE,IL,IL,,60047,(847) 981-8800,07/09/2025,10/22/2025,Yes


In [11]:
print(df["Address"].head(10).tolist())

['4137 W MELROSE ST, CHICAGO, IL 60641', '9325 OSCEOLA AVE, MORTON GROVE, IL 60053', '3336 S LOMBARD, BERWYN, IL 60402', '4532 W SCHUBERT AVE, CHICAGO, IL 60639', '1580 N NORTHWEST HIGHWAY, PARK RIDGE, IL', '5033 N NASHVILLE AVENUE, CHICAGO, IL 60656', '5105 N TRIPP AVE, CHICAGO, IL 60630', 'PO BOX 45, NEW LENOX, IL 60451', '330 BOND STREET, ELK GROVE VILLAGE, IL 60007', '330 BOND ST, ELK GROVE VILLAGE, IL 60047']
